Audyt metodologii 2026-09-10: wyniki historyczne unieważnione. Definicje i ograniczenia: `../docs/methodology_audit.md`. Przeliczenia lokalne: `../data/processed/audit_v2/`.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Środowisko działa")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

parquet_files = list(DATA_RAW.glob("*.parquet"))

print(f"Liczba plików parquet: {len(parquet_files)}")
for f in parquet_files:
    print(f.name)

print(Path.cwd())
list(DATA_RAW.iterdir())
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from src.time_analysis import (validate_time, time_shift, past_mean, lagged_corr,
    rapping_starts, rapping_features, complete_window, tail_mask as post_tail_mask, coverage)


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print("Interim files:")
for f in DATA_INTERIM.glob("*.parquet"):
    print(f.name)

print("\nProcessed files:")
for f in DATA_PROCESSED.glob("*.parquet"):
    print(f.name)

In [ ]:
df = pd.read_parquet(DATA_PROCESSED / "dataset_clean.parquet")
validate_time(df)
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
plt.figure(figsize=(16,5))

df['008A01345'].plot()

plt.title('Dust concentration after ESP')
plt.ylabel('mg/Nm3')
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(16,5))

plt.plot(df.index, df['008A01345'], linewidth=0.5)

plt.title('Dust concentration after ESP')
plt.ylabel('mg/Nm3')
plt.grid(True)

plt.show()

In [ ]:
df['008A01345'].describe()

Histogram dystrybucji targetu

In [ ]:
plt.figure(figsize=(10,5))

df['008A01345'].hist(bins=100)

plt.title('Dust concentration distribution')
plt.xlabel('mg/Nm3')

plt.show()

Korelacje z targetem

In [ ]:
correlations = df.corr(numeric_only=True)['008A01345'].sort_values(ascending=False)

correlations.head(20)

In [ ]:
correlations.tail(20)

In [ ]:
import seaborn as sns
selected_cols = [
    '008A01345',  # pył

    '008A00936',  # moc generatora

    '008A00719',
    '008A00727',  # ciśnienia

    '008A01350',  # O2
    '008A01341',  # SO2
    '008A01343',  # NOx
    '008A01347',  # CO

    '008A02273',
    '008A02274',
    '008A02275',  # moce ESP

    '008A02289',
    '008A02290',
    '008A02291',  # napięcia wtórne

    '008A02267',
    '008A02268',
    '008A02269',  # prądy wtórne
]

corr_matrix = df[selected_cols].corr()

plt.figure(figsize=(14,10))

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('ESP process correlation matrix')

plt.show()

In [ ]:
plt.figure(figsize=(16,5))

df['008A01345'].rolling("10min").mean().plot()

plt.title('Dust concentration - rolling mean (10 min)')
plt.ylabel('mg/Nm3')

plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(16,6))

df['008A02289'].plot(label='1A')
df['008A02290'].plot(label='2A')
df['008A02291'].plot(label='3A')

plt.title('ESP secondary voltages')
plt.ylabel('kVDC')

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
lag_corrs = []

for lag in range(0, 61):
    corr = time_shift(df['008A02289'], lag).corr(df['008A01345'])
    lag_corrs.append(corr)

plt.figure(figsize=(12,5))

plt.plot(range(0,61), lag_corrs)

plt.title('Lag correlation: Voltage 1A vs Dust')
plt.xlabel('Lag [samples]')
plt.ylabel('Correlation')

plt.grid(True)

plt.show()

In [ ]:
model_cols = [
    '008A01345',  # pył target

    '008A00936',  # moc generatora

    '008A00719',
    '008A00727',  # ciśnienia

    '008A01350',  # O2

    '008A02289',
    '008A02290',
    '008A02291',  # napięcia wtórne

    '008A02267',
    '008A02268',
    '008A02269',  # prądy wtórne

    '008A02264',
    '008A02265',
    '008A02266',  # przeskoki
]

model_df = df[model_cols].copy()

print(model_df.shape)

model_df.head()

In [ ]:
lag_features = [
    '008A02289',  # UWT 1A
    '008A02290',
    '008A02291',
]

for col in lag_features:

    model_df[f'{col}_lag_1m'] = time_shift(model_df[col], 6)

    model_df[f'{col}_lag_5m'] = time_shift(model_df[col], 30)

In [ ]:
# Historical dust is allowed only for online nowcasting with past sensor access.
# Current y(t) is excluded for every rolling feature; gaps reset history.
for col in ['008A01345', '008A02289']:
    model_df[f'{col}_rollmean_5m'] = past_mean(model_df[col], 5)


In [ ]:
model_df.head(40)

In [ ]:
model_df = model_df.dropna()

print(model_df.shape)

In [ ]:
target_col = '008A01345'

X = model_df.drop(columns=[target_col])

y = model_df[target_col]

print(X.shape)
print(y.shape)

In [ ]:
split_idx = int(len(model_df) * 0.8)

X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]

y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print(X_train.shape)
print(X_test.shape)

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

In [ ]:
model = XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.3f}")

In [ ]:
plt.figure(figsize=(16,5))

plt.plot(y_test.values[:2000], label='Real')
plt.plot(y_pred[:2000], label='Predicted')

plt.title('Dust prediction')
plt.ylabel('mg/Nm3')

plt.legend()
plt.grid(True)

plt.show()

In [ ]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
})

importance = importance.sort_values(
    by='importance',
    ascending=False
)

importance.head(20)

In [ ]:
plt.figure(figsize=(10,8))

plt.barh(
    importance['feature'][:15][::-1],
    importance['importance'][:15][::-1]
)

plt.title('Feature importance')

plt.show()

## Baseline status after audit
The historical score used a rolling target feature containing y(t) and is invalid as evidence of predictive performance. Recalculate with past-only features. This is online nowcasting with historical dust sensor access. See ../docs/audit_validation.md for the corrected run and its limits.
